<a href="https://colab.research.google.com/github/nisha-s10/Deep-Learning-Lab-AFI524/blob/main/Experiment%2010/Experiment_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1 — Install Required Libraries

In [ ]:
!pip install timm wandb huggingface_hub --quiet

### Step 2 — Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torchvision.models import resnet18
from torch.utils.data import DataLoader, random_split

import timm
import numpy as np
import matplotlib.pyplot as plt
import time

import wandb

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


### Step 3 — Initialize Weights & Biases

In [ ]:
wandb.init(
    project="Experiment-10-ViT-ResNet",
    mode = "offline"
)

### Step 4 — Dataset Preprocessing

In [ ]:
transform_original = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )

])

### Step 5 — Dataset Augmentation

In [ ]:
transform_augmented = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomVerticalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )

])

### Step 6 — Load CIFAR-10 Dataset

In [ ]:
dataset = torchvision.datasets.CIFAR10(

    root="./data",

    train=True,

    download=True,

    transform=transform_original

)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(

    dataset,
    [train_size, val_size, test_size]

)

### Step 7 — Create Augmented Training Dataset

In [ ]:
train_dataset_aug = torchvision.datasets.CIFAR10(

    root="./data",

    train=True,

    download=True,

    transform=transform_augmented

)

### Step 8 — Create DataLoaders

In [ ]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size
)

### Step 9 — Vision Transformer Model (ViT)

In [ ]:
vit_model = timm.create_model(

    "vit_tiny_patch16_224",

    pretrained=False,

    num_classes=10

)

vit_model = vit_model.to(device)

### Step 10 — ResNet-18 Model

In [ ]:
from torchvision.models import resnet18

resnet_model = resnet18(
    weights=None
)

resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    10
)

resnet_model = resnet_model.to(device)

### Step 11 — Loss Functions

In [ ]:
cross_entropy_loss = nn.CrossEntropyLoss()

label_smoothing_loss = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

### Step 12 — Focal Loss Implementation

In [ ]:
class FocalLoss(nn.Module):

    def __init__(
        self,
        gamma=2
    ):
        super().__init__()

        self.gamma = gamma

        self.ce = nn.CrossEntropyLoss()

    def forward(
        self,
        inputs,
        targets
    ):

        ce_loss = self.ce(
            inputs,
            targets
        )

        pt = torch.exp(
            -ce_loss
        )

        focal_loss = (
            (1 - pt) ** self.gamma
        ) * ce_loss

        return focal_loss


focal_loss = FocalLoss()

### Step 13 — Optimizers

In [ ]:
optimizers = {

    "SGD": optim.SGD,

    "RMSprop": optim.RMSprop,

    "Adam": optim.Adam

}

### Step 14 — Evaluation Function

In [ ]:
def evaluate_model(

    model,
    loader,
    loss_fn

):

    model.eval()

    total_loss = 0

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = loss_fn(
                outputs,
                labels
            )

            total_loss += loss.item()

            _, predicted = torch.max(
                outputs,
                1
            )

            correct += (
                predicted == labels
            ).sum().item()

            total += labels.size(0)

    avg_loss = total_loss / len(loader)

    accuracy = correct / total

    return avg_loss, accuracy

### Step 15 — Training Function

In [ ]:
def train_model(

    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer_class,
    model_name,
    loss_name,
    opt_name

):

    optimizer = optimizer_class(
        model.parameters(),
        lr=0.001
    )

    epochs = 5

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        correct = 0
        total = 0

        start_time = time.time()

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = loss_fn(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

            _, predicted = torch.max(
                outputs,
                1
            )

            correct += (
                predicted == labels
            ).sum().item()

            total += labels.size(0)

        train_loss = total_loss / len(train_loader)

        train_accuracy = correct / total

        val_loss, val_accuracy = evaluate_model(

            model,
            val_loader,
            loss_fn

        )

        epoch_time = time.time() - start_time

        wandb.log({

            "Model": model_name,

            "Loss Function": loss_name,

            "Optimizer": opt_name,

            "Epoch": epoch + 1,

            "Train Loss": train_loss,

            "Validation Loss": val_loss,

            "Train Accuracy": train_accuracy,

            "Validation Accuracy": val_accuracy,

            "Training Time": epoch_time

        })

        print(
            f"Model: {model_name} | "
            f"Loss: {loss_name} | "
            f"Optimizer: {opt_name} | "
            f"Epoch: {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )

### Step 16 — Run Training for Both Models

In [ ]:
loss_functions = {

    "CrossEntropy": cross_entropy_loss,

    "LabelSmoothing": label_smoothing_loss,

    "FocalLoss": focal_loss

}

for loss_name, loss_fn in loss_functions.items():

    for opt_name, opt_class in optimizers.items():

        print(
            f"\nTraining ViT | {loss_name} | {opt_name}"
        )

        # Create NEW ViT model each time

        vit_model = timm.create_model(
            "vit_tiny_patch16_224",
            pretrained=False,
            num_classes=10
        ).to(device)

        train_model(

            vit_model,

            train_loader,

            val_loader,

            loss_fn,

            opt_class,

            "ViT",

            loss_name,

            opt_name

        )

        print(
            f"\nTraining ResNet | {loss_name} | {opt_name}"
        )

        # Create NEW ResNet model each time

        resnet_model = resnet18(
            weights=None
        )

        resnet_model.fc = nn.Linear(
            resnet_model.fc.in_features,
            10
        )

        resnet_model = resnet_model.to(device)

        train_model(

            resnet_model,

            train_loader,

            val_loader,

            loss_fn,

            opt_class,

            "ResNet18",

            loss_name,

            opt_name

        )


Training ViT | CrossEntropy | SGD
Model: ViT | Loss: CrossEntropy | Optimizer: SGD | Epoch: 1/5 | Train Loss: 1.9551 | Val Loss: 1.8622 | Val Acc: 0.3134


### Step 17 — Train on Augmented Dataset

In [ ]:
print("Training with Augmented Dataset")

train_model(

    vit_model,

    train_loader_aug,

    val_loader,

    cross_entropy_loss,

    optim.Adam,

    "ViT Augmented",

    "CrossEntropy",

    "Adam"

)

train_model(

    resnet_model,

    train_loader_aug,

    val_loader,

    cross_entropy_loss,

    optim.Adam,

    "ResNet Augmented",

    "CrossEntropy",

    "Adam"

)

### Step 18 — Final Test Evaluation

In [ ]:
vit_test_loss, vit_test_acc = evaluate_model(

    vit_model,

    test_loader,

    cross_entropy_loss

)

resnet_test_loss, resnet_test_acc = evaluate_model(

    resnet_model,

    test_loader,

    cross_entropy_loss

)

print(
    "ViT Test Accuracy:",
    vit_test_acc
)

print(
    "ResNet Test Accuracy:",
    resnet_test_acc
)

### Step 19 — Save Models

In [ ]:
torch.save(

    vit_model.state_dict(),

    "vit_model.pth"

)

torch.save(

    resnet_model.state_dict(),

    "resnet18_model.pth"

)

print("Models saved")